In [1]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import TargetEncoder

print("Импорты выполнены")

Импорты выполнены


## Пути и загрзуки данных

In [2]:
import sys
import numpy as np
import pandas as pd

print(sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
numpy: 2.4.3
pandas: 2.2.2


In [3]:
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "clean_dataset.pkl"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)

df = pd.read_pickle(DATA_PATH)
print(df.shape)
df.head()

PROJECT_ROOT = C:\Users\79022\Desktop\iis\iis
DATA_PATH = C:\Users\79022\Desktop\iis\iis\data\clean_dataset.pkl
(2000, 21)


,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [4]:
TARGET_COL = "price_range"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Классы:", sorted(y.unique()))

X shape: (2000, 20)
y shape: (2000,)
Классы: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1500, 20)
X_test: (500, 20)
y_train: (1500,)
y_test: (500,)


In [6]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)
print("Количество числовых:", len(numeric_features))
print("Количество категориальных:", len(categorical_features))

numeric_features: ['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g', 'touch_screen', 'wifi']
categorical_features: []
Количество числовых: 20
Количество категориальных: 0


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", TargetEncoder(target_type="multiclass"), categorical_features),
    ],
    remainder="drop",
)

In [8]:
baseline_pipeline = Pipeline(
    steps=[
        ("transform", preprocessor),
        (
            "classification",
            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        ),
    ]
)

baseline_pipeline

Pipeline(steps=[('transform',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['battery_power', 'blue',
                                                   'clock_speed', 'dual_sim',
                                                   'fc', 'four_g', 'int_memory',
                                                   'm_dep', 'mobile_wt',
                                                   'n_cores', 'pc', 'px_height',
                                                   'px_width', 'ram', 'sc_h',
                                                   'sc_w', 'talk_time',
                                                   'three_g', 'touch_screen',
                                                   'wifi']),
                                                 ('cat',
                                                  TargetEncoder(target_type='multiclass'),
                                                  [])])),
                ('classification',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

In [9]:
baseline_pipeline.fit(X_train, y_train)

print("Baseline model trained")

Baseline model trained


In [34]:
y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)

metrics = {
    "precision_weighted": precision_score(y_test, y_pred, average="weighted"),
    "recall_weighted": recall_score(y_test, y_pred, average="weighted"),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
    "roc_auc_ovr_weighted": roc_auc_score(
        y_test, y_proba, multi_class="ovr", average="weighted"
    ),
}

metrics_df = pd.DataFrame(
    {"metric": list(metrics.keys()), "value": list(metrics.values())}
).sort_values("value", ascending=False)

metrics_df

,metric,value
3,roc_auc_ovr_weighted,0.977581
0,precision_weighted,0.874530
2,f1_weighted,0.874205
1,recall_weighted,0.874000


## Подготовка артефактов для MLFlow

In [10]:
from mlflow.models import infer_signature

input_example = X_train.head(5)
signature = infer_signature(
    model_input=X_train.head(5), model_output=baseline_pipeline.predict(X_train.head(5))
)

print("Input example shape:", input_example.shape)
signature

Input example shape: (5, 20)


inputs: 
  ['battery_power': long (required), 'blue': long (required), 'clock_speed': double (required), 'dual_sim': long (required), 'fc': long (required), 'four_g': long (required), 'int_memory': long (required), 'm_dep': double (required), 'mobile_wt': long (required), 'n_cores': long (required), 'pc': long (required), 'px_height': long (required), 'px_width': long (required), 'ram': long (required), 'sc_h': long (required), 'sc_w': long (required), 'talk_time': long (required), 'three_g': long (required), 'touch_screen': long (required), 'wifi': long (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

## Сохранение baseline-метрик локально

In [11]:
ARTIFACTS_DIR = PROJECT_ROOT / "research" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_metrics_path = ARTIFACTS_DIR / "baseline_metrics.json"

with open(baseline_metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)

print("Сохранено:", baseline_metrics_path)

Сохранено: C:\Users\79022\Desktop\iis\iis\research\artifacts\baseline_metrics.json


## Заготовка под MLflow logging

In [10]:
import mlflow
import mlflow.sklearn

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "mobile_price_lab2"
RUN_NAME = "baseline_random_forest"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

Tracking URI: http://127.0.0.1:5000
Experiment: mobile_price_lab2


## Логирование baseline модели

In [ ]:
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "n_estimators": 100,
            "random_state": 42,
            "test_size": 0.25,
            "target_column": TARGET_COL,
            "numeric_features_count": len(numeric_features),
            "categorical_features_count": len(categorical_features),
        }
    )

    mlflow.log_metrics(metrics)

    mlflow.log_artifact(str(baseline_metrics_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=baseline_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="mobile_price_classifier",
    )

print("Baseline run logged to MLflow")

Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
2026/03/09 12:57:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_classifier, version 2
Created version '2' of model 'mobile_price_classifier'.


2026/03/09 12:57:25 INFO mlflow.tracking._tracking_service.client: 🏃 View run sklearn_feature_engineering_random_forest at: http://127.0.0.1:5000/#/experiments/188674586182771687/runs/e3cb2556b2f64a6aa6e566934ea962da.
2026/03/09 12:57:25 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/188674586182771687.


Baseline run logged to MLflow


In [16]:
from sklearn.preprocessing import (
    PolynomialFeatures,
    KBinsDiscretizer,
    FunctionTransformer,
)
from sklearn.base import BaseEstimator, TransformerMixin

## Выбираем признаки для генерации новых

In [38]:
poly_features = ["ram", "battery_power"]
bin_features = ["px_height", "px_width", "int_memory"]

print("poly_features:", poly_features)
print("bin_features:", bin_features)

poly_features: ['ram', 'battery_power']
bin_features: ['px_height', 'px_width', 'int_memory']


## Подготовка списка признаков

In [39]:
base_numeric_features = [
    col for col in numeric_features if col not in poly_features + bin_features
]

print("base_numeric_features:", len(base_numeric_features))
print("poly_features:", poly_features)
print("bin_features:", bin_features)

base_numeric_features: 15
poly_features: ['ram', 'battery_power']
bin_features: ['px_height', 'px_width', 'int_memory']


## ColumnTransformer с новыми признаками
### Полиномиальные признаки по заданию тоже должны масштабироваться

In [40]:
fe_preprocessor = ColumnTransformer(
    transformers=[
        ("num_base", StandardScaler(), base_numeric_features),
        (
            "poly",
            Pipeline(
                [
                    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                    ("scaler", StandardScaler()),
                ]
            ),
            poly_features,
        ),
        (
            "bins",
            Pipeline(
                [
                    (
                        "kbins",
                        KBinsDiscretizer(
                            n_bins=4, encode="onehot-dense", strategy="quantile"
                        ),
                    )
                ]
            ),
            bin_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

fe_preprocessor

ColumnTransformer(transformers=[('num_base', StandardScaler(),
                                 ['blue', 'clock_speed', 'dual_sim', 'fc',
                                  'four_g', 'm_dep', 'mobile_wt', 'n_cores',
                                  'pc', 'sc_h', 'sc_w', 'talk_time', 'three_g',
                                  'touch_screen', 'wifi']),
                                ('poly',
                                 Pipeline(steps=[('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('scaler', StandardScaler())]),
                                 ['ram', 'battery_power']),
                                ('bins',
                                 Pipeline(steps=[('kbins',
                                                  KBinsDiscretizer(encode='onehot-dense',
                                                                   n_bins=4))]),
                                 ['px_height', 'px_width', 'int_memory'])],
                  verbose_feature_names_out=False)

In [41]:
X_train_fe_sklearn = X_train.copy()

X_train_fe_sklearn_transformed = fe_preprocessor.fit_transform(
    X_train_fe_sklearn, y_train
)
X_test_fe_sklearn_transformed = fe_preprocessor.transform(X_test)

feature_names_sklearn = fe_preprocessor.get_feature_names_out()

X_train_fe_sklearn_df = pd.DataFrame(
    X_train_fe_sklearn_transformed, columns=feature_names_sklearn, index=X_train.index
)

X_test_fe_sklearn_df = pd.DataFrame(
    X_test_fe_sklearn_transformed, columns=feature_names_sklearn, index=X_test.index
)

print("train FE shape:", X_train_fe_sklearn_df.shape)
print("test FE shape:", X_test_fe_sklearn_df.shape)
X_train_fe_sklearn_df.head()

train FE shape: (1500, 32)
test FE shape: (500, 32)


,blue,clock_speed,dual_sim,fc,four_g,m_dep,mobile_wt,n_cores,pc,sc_h,...,px_height_2.0,px_height_3.0,px_width_0.0,px_width_1.0,px_width_2.0,px_width_3.0,int_memory_0.0,int_memory_1.0,int_memory_2.0,int_memory_3.0
623,1.001334,-0.272000,0.989390,0.625404,0.965914,-0.013338,-0.738036,-1.550579,1.007810,-0.781520,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
822,-0.998668,0.581556,0.989390,-0.996816,-1.035289,-0.358292,0.985730,1.072581,-1.136233,0.649253,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1736,1.001334,1.313176,0.989390,-0.533324,-1.035289,1.021524,-1.416238,-1.550579,-1.136233,-0.066134,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
474,-0.998668,-0.759746,-1.010724,-0.996816,-1.035289,1.021524,-0.709777,-0.676192,-0.476527,-0.781520,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
561,-0.998668,-1.247493,0.989390,2.479370,-1.035289,1.021524,1.437866,-1.113386,1.007810,1.126177,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


## Сохранение названий новых столбцов

In [40]:
ARTIFACTS_DIR = PROJECT_ROOT / "research" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

sklearn_feature_names_path = ARTIFACTS_DIR / "sklearn_feature_names.txt"

with open(sklearn_feature_names_path, "w", encoding="utf-8") as f:
    for col in feature_names_sklearn:
        f.write(f"{col}\n")

print("Сохранено:", sklearn_feature_names_path)
print("Количество признаков после FE:", len(feature_names_sklearn))

Сохранено: C:\Users\79022\Desktop\iis\iis\research\artifacts\sklearn_feature_names.txt
Количество признаков после FE: 32


## Pipeline с новым preprocessor

In [37]:
fe_pipeline = Pipeline(
    steps=[
        ("transform", fe_preprocessor),
        (
            "classification",
            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        ),
    ]
)

fe_pipeline

Pipeline(steps=[('transform',
                 ColumnTransformer(transformers=[('num_base', StandardScaler(),
                                                  ['blue', 'clock_speed',
                                                   'dual_sim', 'fc', 'four_g',
                                                   'm_dep', 'mobile_wt',
                                                   'n_cores', 'pc', 'sc_h',
                                                   'sc_w', 'talk_time',
                                                   'three_g', 'touch_screen',
                                                   'wifi']),
                                                 ('poly',
                                                  Pipeline(steps=[('poly',
                                                                   PolynomialFeatures(include_bias=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['ram', 'battery_power']),
                                                 ('bins',
                                                  Pipeline(steps=[('kbins',
                                                                   KBinsDiscretizer(encode='onehot-dense',
                                                                                    n_bins=4))]),
                                                  ['px_height', 'px_width',
                                                   'int_memory'])],
                                   verbose_feature_names_out=False)),
                ('classification',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

## Обучение и метрики

In [42]:
fe_pipeline.fit(X_train, y_train)

y_pred_fe = fe_pipeline.predict(X_test)
y_proba_fe = fe_pipeline.predict_proba(X_test)

fe_metrics = {
    "precision_weighted": precision_score(y_test, y_pred_fe, average="weighted"),
    "recall_weighted": recall_score(y_test, y_pred_fe, average="weighted"),
    "f1_weighted": f1_score(y_test, y_pred_fe, average="weighted"),
    "roc_auc_ovr_weighted": roc_auc_score(
        y_test, y_proba_fe, multi_class="ovr", average="weighted"
    ),
}

pd.DataFrame(
    {"metric": list(fe_metrics.keys()), "value": list(fe_metrics.values())}
).sort_values("value", ascending=False)

,metric,value
3,roc_auc_ovr_weighted,0.982163
1,recall_weighted,0.884000
2,f1_weighted,0.883871
0,precision_weighted,0.883756


## Сравнение с baseline

In [43]:
comparison_df = pd.DataFrame(
    [
        {"model": "baseline", **metrics},
        {"model": "sklearn_feature_engineering", **fe_metrics},
    ]
)

comparison_df

,model,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_weighted
0,baseline,0.874530,0.874,0.874205,0.977581
1,sklearn_feature_engineering,0.883756,0.884,0.883871,0.982163


## Подготовка signature и input_example

In [44]:
from mlflow.models import infer_signature

fe_input_example = X_train.head(5)
fe_signature = infer_signature(
    model_input=X_train.head(5), model_output=fe_pipeline.predict(X_train.head(5))
)

fe_signature

inputs: 
  ['battery_power': long (required), 'blue': long (required), 'clock_speed': double (required), 'dual_sim': long (required), 'fc': long (required), 'four_g': long (required), 'int_memory': long (required), 'm_dep': double (required), 'mobile_wt': long (required), 'n_cores': long (required), 'pc': long (required), 'px_height': long (required), 'px_width': long (required), 'ram': long (required), 'sc_h': long (required), 'sc_w': long (required), 'talk_time': long (required), 'three_g': long (required), 'touch_screen': long (required), 'wifi': long (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

## Сохранение метрики в Json

In [45]:
fe_metrics_path = ARTIFACTS_DIR / "sklearn_fe_metrics.json"

with open(fe_metrics_path, "w", encoding="utf-8") as f:
    json.dump(fe_metrics, f, ensure_ascii=False, indent=4)

print("Сохранено:", fe_metrics_path)

Сохранено: C:\Users\79022\Desktop\iis\iis\research\artifacts\sklearn_fe_metrics.json



## Логирование второго run в MLfLOW

In [47]:
RUN_NAME = "sklearn_feature_engineering_random_forest"

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "feature_stage": "sklearn_feature_engineering",
            "n_estimators": 100,
            "random_state": 42,
            "poly_features": ", ".join(poly_features),
            "bin_features": ", ".join(bin_features),
            "poly_degree": 2,
            "kbins_n_bins": 4,
            "kbins_strategy": "quantile",
            "features_after_fe": len(feature_names_sklearn),
        }
    )

    mlflow.log_metrics(fe_metrics)

    mlflow.log_artifact(str(fe_metrics_path))
    mlflow.log_artifact(str(sklearn_feature_names_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=fe_pipeline,
        artifact_path="model",
        signature=fe_signature,
        input_example=fe_input_example,
        registered_model_name="mobile_price_classifier",
    )

print("Sklearn FE run logged to MLflow")

Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
2026/03/09 13:07:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_classifier, version 3
Created version '3' of model 'mobile_price_classifier'.


2026/03/09 13:07:33 INFO mlflow.tracking._tracking_service.client: 🏃 View run sklearn_feature_engineering_random_forest at: http://127.0.0.1:5000/#/experiments/188674586182771687/runs/84d1c666f65c4fa69f143e0cdeb64270.
2026/03/09 13:07:33 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/188674586182771687.


Sklearn FE run logged to MLflow


## Выбираем количество признаков N

In [1]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

In [22]:
print("X_train_fe_sklearn_df shape:", X_train_fe_sklearn_df.shape)
print("X_test_fe_sklearn_df shape:", X_test_fe_sklearn_df.shape)
print("Количество признаков после FE:", X_train_fe_sklearn_df.shape[1])
total_features = X_train_fe_sklearn_df.shape[1]
N_FEATURES = max(1, int(total_features * 0.5))

print("Всего признаков:", total_features)
print("Будем отбирать N =", N_FEATURES)

X_train_fe_sklearn_df shape: (1500, 32)
X_test_fe_sklearn_df shape: (500, 32)
Количество признаков после FE: 32
Всего признаков: 32
Будем отбирать N = 16


## Обучаем SequentialFeatureSelector

In [ ]:
if not hasattr(np, "NINF"):
    np.NINF = -np.inf
sfs_estimator = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

sfs = SFS(
    sfs_estimator,
    k_features=N_FEATURES,
    forward=True,
    floating=False,
    scoring="f1_weighted",
    cv=3,
    n_jobs=-1,
)

sfs = sfs.fit(X_train_fe_sklearn_df, y_train)

print("SFS finished")


SFS finished


## Получить индексы и названия выбранных признаков

In [25]:
selected_feature_idx = list(sfs.k_feature_idx_)
selected_feature_names = list(sfs.k_feature_names_)

print("Количество выбранных признаков:", len(selected_feature_idx))
print("Индексы:", selected_feature_idx[:10], "...")
print("Названия:", selected_feature_names[:10], "...")

Количество выбранных признаков: 16
Индексы: [7, 9, 13, 17, 18, 19, 20, 21, 22, 23] ...
Названия: ['n_cores', 'sc_h', 'touch_screen', 'ram^2', 'ram battery_power', 'battery_power^2', 'px_height_0.0', 'px_height_1.0', 'px_height_2.0', 'px_height_3.0'] ...


## Сохранение выбранных признаков и названий

In [27]:
ARTIFACTS_DIR = PROJECT_ROOT / "research" / "artifacts"
selected_feature_idx_path = ARTIFACTS_DIR / "selected_feature_indices.txt"
selected_feature_names_path = ARTIFACTS_DIR / "selected_feature_names.txt"

with open(selected_feature_idx_path, "w", encoding="utf-8") as f:
    for idx in selected_feature_idx:
        f.write(f"{idx}\n")

with open(selected_feature_names_path, "w", encoding="utf-8") as f:
    for name in selected_feature_names:
        f.write(f"{name}\n")

print("Сохранены файлы:")
print(selected_feature_idx_path)
print(selected_feature_names_path)

Сохранены файлы:
C:\Users\79022\Desktop\iis\iis\research\artifacts\selected_feature_indices.txt
C:\Users\79022\Desktop\iis\iis\research\artifacts\selected_feature_names.txt


## Сформируем train/test только с выбранными признаками

In [28]:
X_train_selected_df = X_train_fe_sklearn_df.iloc[:, selected_feature_idx].copy()
X_test_selected_df = X_test_fe_sklearn_df.iloc[:, selected_feature_idx].copy()

print("X_train_selected_df:", X_train_selected_df.shape)
print("X_test_selected_df:", X_test_selected_df.shape)
display(X_train_selected_df.head())

X_train_selected_df: (1500, 16)
X_test_selected_df: (500, 16)


,n_cores,sc_h,touch_screen,ram^2,ram battery_power,battery_power^2,px_height_0.0,px_height_1.0,px_height_2.0,px_height_3.0,px_width_0.0,px_width_1.0,px_width_3.0,int_memory_0.0,int_memory_1.0,int_memory_3.0
623,-1.550579,-0.781520,-1.016130,-0.000536,-0.636644,-1.184338,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
822,1.072581,0.649253,-1.016130,1.904341,0.344824,-0.922836,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1736,-1.550579,-0.066134,0.984126,-0.869628,-0.238692,1.302500,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
474,-0.676192,-0.781520,-1.016130,-0.059073,1.134226,1.942889,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
561,-1.113386,1.126177,-1.016130,-0.611019,-0.336568,-0.193524,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [ ]:
fs_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)

fs_model.fit(X_train_selected_df, y_train)

y_pred_fs = fs_model.predict(X_test_selected_df)
y_proba_fs = fs_model.predict_proba(X_test_selected_df)

In [ ]:
fs_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)

fs_model.fit(X_train_selected_df, y_train)

y_pred_fs = fs_model.predict(X_test_selected_df)
y_proba_fs = fs_model.predict_proba(X_test_selected_df)


In [ ]:
fs_metrics = {
    "precision_weighted": precision_score(y_test, y_pred_fs, average="weighted"),
    "recall_weighted": recall_score(y_test, y_pred_fs, average="weighted"),
    "f1_weighted": f1_score(y_test, y_pred_fs, average="weighted"),
    "roc_auc_ovr_weighted": roc_auc_score(
        y_test, y_proba_fs, multi_class="ovr", average="weighted"
    ),
}

pd.DataFrame(
    {"metric": list(fs_metrics.keys()), "value": list(fs_metrics.values())}
).sort_values("value", ascending=False)

,metric,value
3,roc_auc_ovr_weighted,0.988509
0,precision_weighted,0.915295
2,f1_weighted,0.914424
1,recall_weighted,0.914000


## Сравним SRS с baseline и FE

In [ ]:
comparison_df = pd.DataFrame(
    [
        {"model": "baseline", **metrics},
        {"model": "sklearn_feature_engineering", **fe_metrics},
        {"model": "sklearn_fe + mlxtend_sfs", **fs_metrics},
    ]
)

comparison_df

,model,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_weighted
0,baseline,0.874530,0.874,0.874205,0.977581
1,sklearn_feature_engineering,0.883756,0.884,0.883871,0.982163
2,sklearn_fe + mlxtend_sfs,0.915295,0.914,0.914424,0.988509


In [ ]:
fs_metrics_path = ARTIFACTS_DIR / "mlxtend_sfs_metrics.json"

with open(fs_metrics_path, "w", encoding="utf-8") as f:
    json.dump(fs_metrics, f, ensure_ascii=False, indent=4)

print("Сохранено:", fs_metrics_path)

Сохранено: C:\Users\79022\Desktop\iis\iis\research\artifacts\mlxtend_sfs_metrics.json


In [ ]:
from mlflow.models import infer_signature

fs_input_example = X_train_selected_df.head(5)
fs_signature = infer_signature(
    model_input=X_train_selected_df.head(5),
    model_output=fs_model.predict(X_train_selected_df.head(5)),
)

fs_signature

inputs: 
  ['n_cores': double (required), 'sc_h': double (required), 'touch_screen': double (required), 'ram^2': double (required), 'ram battery_power': double (required), 'battery_power^2': double (required), 'px_height_0.0': double (required), 'px_height_1.0': double (required), 'px_height_2.0': double (required), 'px_height_3.0': double (required), 'px_width_0.0': double (required), 'px_width_1.0': double (required), 'px_width_3.0': double (required), 'int_memory_0.0': double (required), 'int_memory_1.0': double (required), 'int_memory_3.0': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

In [ ]:
RUN_NAME = "mlxtend_forward_selection_random_forest"

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "feature_stage": "mlxtend_forward_selection",
            "n_estimators": 100,
            "random_state": 42,
            "total_features_before_selection": total_features,
            "selected_features_count": len(selected_feature_idx),
            "selection_direction": "forward",
            "selection_cv": 3,
            "selection_metric": "f1_weighted",
        }
    )

    mlflow.log_metrics(fs_metrics)

    mlflow.log_artifact(str(fs_metrics_path))
    mlflow.log_artifact(str(selected_feature_idx_path))
    mlflow.log_artifact(str(selected_feature_names_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=fs_model,
        artifact_path="model",
        signature=fs_signature,
        input_example=fs_input_example,
        registered_model_name="mobile_price_classifier",
    )

print("MLxtend SFS run logged to MLflow")

Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
2026/03/09 14:24:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_classifier, version 4
Created version '4' of model 'mobile_price_classifier'.


2026/03/09 14:24:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run mlxtend_forward_selection_random_forest at: http://127.0.0.1:5000/#/experiments/188674586182771687/runs/6d2e1a382dc94aa0b3d82f3d30df7280.
2026/03/09 14:24:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/188674586182771687.


MLxtend SFS run logged to MLflow


In [48]:
X_train_optuna = X_train_selected_df.copy()
X_test_optuna = X_test_selected_df.copy()

print("X_train_optuna:", X_train_optuna.shape)
print("X_test_optuna:", X_test_optuna.shape)

X_train_optuna: (1500, 16)
X_test_optuna: (500, 16)


In [49]:
import optuna

In [ ]:
# Определи objective function
# По ЛР для классификации настраиваем по f1, значит используем f1_weighted. Направление оптимизации будет maximize
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        random_state=42,
        n_jobs=1,
    )

    model.fit(X_train_optuna, y_train)
    y_pred = model.predict(X_test_optuna)

    f1 = f1_score(y_test, y_pred, average="weighted")
    return f1

## Запустим исследованием минимум 10 trials 


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
print("Best value (f1_weighted):", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-09 14:40:14,493] A new study created in memory with name: no-name-c81649e3-92a1-47e0-be27-dce1d67c72ca
[I 2026-03-09 14:40:15,406] Trial 0 finished with value: 0.9024603685550964 and parameters: {'n_estimators': 88, 'max_depth': 15, 'max_features': 0.9068287516082584}. Best is trial 0 with value: 0.9024603685550964.
[I 2026-03-09 14:40:16,371] Trial 1 finished with value: 0.8965092447966886 and parameters: {'n_estimators': 169, 'max_depth': 10, 'max_features': 0.8748221406672637}. Best is trial 0 with value: 0.9024603685550964.
[I 2026-03-09 14:40:17,517] Trial 2 finished with value: 0.9065997471288626 and parameters: {'n_estimators': 241, 'max_depth': 10, 'max_features': 0.6829568649210176}. Best is trial 2 with value: 0.9065997471288626.
[I 2026-03-09 14:40:18,409] Trial 3 finished with value: 0.9043771301291358 and parameters: {'n_estimators': 240, 'max_depth': 18, 'max_features': 0.3766268359120737}. Best is trial 2 with value: 0.9065997471288626.
[I 2026-03-09 14:40:19,

Best trial:
Best value (f1_weighted): 0.9124789069472393
Best params: {'n_estimators': 134, 'max_depth': 17, 'max_features': 0.49677913225313786}


## Обучим лучшую модель

In [ ]:
best_params = study.best_params

optuna_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=1)

optuna_model.fit(X_train_optuna, y_train)

y_pred_optuna = optuna_model.predict(X_test_optuna)
y_proba_optuna = optuna_model.predict_proba(X_test_optuna)

## Посчитаем метрики

In [ ]:
optuna_metrics = {
    "precision_weighted": precision_score(y_test, y_pred_optuna, average="weighted"),
    "recall_weighted": recall_score(y_test, y_pred_optuna, average="weighted"),
    "f1_weighted": f1_score(y_test, y_pred_optuna, average="weighted"),
    "roc_auc_ovr_weighted": roc_auc_score(
        y_test, y_proba_optuna, multi_class="ovr", average="weighted"
    ),
}

pd.DataFrame(
    {"metric": list(optuna_metrics.keys()), "value": list(optuna_metrics.values())}
).sort_values("value", ascending=False)

,metric,value
3,roc_auc_ovr_weighted,0.989123
0,precision_weighted,0.913148
2,f1_weighted,0.912479
1,recall_weighted,0.912000


## Сравним все этапы



In [ ]:
comparison_df = pd.DataFrame(
    [
        {"model": "baseline", **metrics},
        {"model": "sklearn_feature_engineering", **fe_metrics},
        {"model": "sklearn_fe + mlxtend_sfs", **fs_metrics},
        {"model": "optuna_tuned_best_model", **optuna_metrics},
    ]
)

comparison_df.sort_values("f1_weighted", ascending=False)


,model,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_weighted
2,sklearn_fe + mlxtend_sfs,0.915295,0.914,0.914424,0.988509
3,optuna_tuned_best_model,0.913148,0.912,0.912479,0.989123
1,sklearn_feature_engineering,0.883756,0.884,0.883871,0.982163
0,baseline,0.874530,0.874,0.874205,0.977581


In [58]:
optuna_metrics_path = ARTIFACTS_DIR / "optuna_metrics.json"
optuna_best_params_path = ARTIFACTS_DIR / "optuna_best_params.json"

with open(optuna_metrics_path, "w", encoding="utf-8") as f:
    json.dump(optuna_metrics, f, ensure_ascii=False, indent=4)

with open(optuna_best_params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, ensure_ascii=False, indent=4)

print("Сохранено:")
print(optuna_metrics_path)
print(optuna_best_params_path)

Сохранено:
C:\Users\79022\Desktop\iis\iis\research\artifacts\optuna_metrics.json
C:\Users\79022\Desktop\iis\iis\research\artifacts\optuna_best_params.json


In [ ]:
optuna_input_example = X_train_optuna.head(5)
optuna_signature = infer_signature(
    model_input=X_train_optuna.head(5),
    model_output=optuna_model.predict(X_train_optuna.head(5)),
)

optuna_signature

inputs: 
  ['n_cores': double (required), 'sc_h': double (required), 'touch_screen': double (required), 'ram^2': double (required), 'ram battery_power': double (required), 'battery_power^2': double (required), 'px_height_0.0': double (required), 'px_height_1.0': double (required), 'px_height_2.0': double (required), 'px_height_3.0': double (required), 'px_width_0.0': double (required), 'px_width_1.0': double (required), 'px_width_3.0': double (required), 'int_memory_0.0': double (required), 'int_memory_1.0': double (required), 'int_memory_3.0': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

In [ ]:
RUN_NAME = "optuna_random_forest_best_model"

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "feature_stage": "optuna_tuning",
            "source_features": "selected_features_from_mlxtend",
            "n_trials": 10,
            "optimization_metric": "f1_weighted",
            "optimization_direction": "maximize",
            **best_params,
        }
    )

    mlflow.log_metrics(optuna_metrics)

    mlflow.log_artifact(str(optuna_metrics_path))
    mlflow.log_artifact(str(optuna_best_params_path))
    mlflow.log_artifact(str(selected_feature_idx_path))
    mlflow.log_artifact(str(selected_feature_names_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=optuna_model,
        artifact_path="model",
        signature=optuna_signature,
        input_example=optuna_input_example,
        registered_model_name="mobile_price_classifier",
    )

print("Optuna run logged to MLflow")


Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
2026/03/09 15:12:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_classifier, version 5
Created version '5' of model 'mobile_price_classifier'.


2026/03/09 15:12:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run optuna_random_forest_best_model at: http://127.0.0.1:5000/#/experiments/188674586182771687/runs/cd61c098d74b46d2973f554c2cae6510.
2026/03/09 15:12:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/188674586182771687.


Optuna run logged to MLflow


In [61]:
comparison_df.sort_values("f1_weighted", ascending=False)

,model,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_weighted
2,sklearn_fe + mlxtend_sfs,0.915295,0.914,0.914424,0.988509
3,optuna_tuned_best_model,0.913148,0.912,0.912479,0.989123
1,sklearn_feature_engineering,0.883756,0.884,0.883871,0.982163
0,baseline,0.874530,0.874,0.874205,0.977581


п.16

In [62]:
# Лучшей оказалась модель под названием sklearn_fe + mlxtend_sfs

In [64]:
# Готовим полный датасет
X_full = X.copy()
y_full = y.copy()

In [ ]:
# Применить feature engineering ко всей выборке
X_full_fe = fe_preprocessor.fit_transform(X_full, y_full)
feature_names_full = fe_preprocessor.get_feature_names_out()

X_full_fe_df = pd.DataFrame(X_full_fe, columns=feature_names_full, index=X_full.index)

print("X_full_fe_df:", X_full_fe_df.shape)
display(X_full_fe_df.head())

X_full_fe_df: (2000, 32)


,blue,clock_speed,dual_sim,fc,four_g,m_dep,mobile_wt,n_cores,pc,sc_h,...,px_height_2.0,px_height_3.0,px_width_0.0,px_width_1.0,px_width_2.0,px_width_3.0,int_memory_0.0,int_memory_1.0,int_memory_2.0,int_memory_3.0
0,-0.990050,0.830779,-1.019184,-0.762495,-1.043966,0.340740,1.349249,-1.101971,-1.305750,-0.784983,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.010051,-1.253064,0.981177,-0.992890,0.957886,0.687548,-0.120059,-0.664768,-0.645989,1.114266,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,1.010051,-1.253064,0.981177,-0.532099,0.957886,1.381165,0.134244,0.209639,-0.645989,-0.310171,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.010051,1.198517,-1.019184,-0.992890,-1.043966,1.034357,-0.261339,0.646842,-0.151168,0.876859,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,1.010051,-0.395011,-1.019184,2.002254,0.957886,0.340740,0.021220,-1.101971,0.673534,-1.022389,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [66]:
# Оставить только признаки, выбранные через SFS
X_full_selected_df = X_full_fe_df.iloc[:, selected_feature_idx].copy()

print("X_full_selected_df:", X_full_selected_df.shape)
display(X_full_selected_df.head())

X_full_selected_df: (2000, 16)


,n_cores,sc_h,touch_screen,ram^2,ram battery_power,battery_power^2,px_height_0.0,px_height_1.0,px_height_2.0,px_height_3.0,px_width_0.0,px_width_1.0,px_width_3.0,int_memory_0.0,int_memory_1.0,int_memory_3.0
0,-1.101971,-0.784983,-1.006018,0.170884,-0.282414,-0.919195,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.664768,1.114266,0.994018,0.260599,0.032471,-0.618071,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.209639,-0.310171,0.994018,0.229645,-0.679389,-1.273159,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.646842,0.876859,-1.006018,0.417994,-0.540928,-1.217846,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
4,-1.101971,-1.022389,0.994018,-0.780940,-0.035650,1.434946,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
# Обучаем production-модель на всей выборке
production_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)

production_model.fit(X_full_selected_df, y_full)

print("Production model trained on full dataset")

Production model trained on full dataset


In [68]:
# Сохраняем список используемых признаков
production_feature_names_path = ARTIFACTS_DIR / "production_feature_names.txt"

with open(production_feature_names_path, "w", encoding="utf-8") as f:
    for col in X_full_selected_df.columns:
        f.write(f"{col}\n")

print("Сохранено:", production_feature_names_path)

Сохранено: C:\Users\79022\Desktop\iis\iis\research\artifacts\production_feature_names.txt


In [ ]:
## Готовим signature и input_example

from mlflow.models import infer_signature

production_input_example = X_full_selected_df.head(5)
production_signature = infer_signature(
    model_input=X_full_selected_df.head(5),
    model_output=production_model.predict(X_full_selected_df.head(5)),
)

production_signature


inputs: 
  ['n_cores': double (required), 'sc_h': double (required), 'touch_screen': double (required), 'ram^2': double (required), 'ram battery_power': double (required), 'battery_power^2': double (required), 'px_height_0.0': double (required), 'px_height_1.0': double (required), 'px_height_2.0': double (required), 'px_height_3.0': double (required), 'px_width_0.0': double (required), 'px_width_1.0': double (required), 'px_width_3.0': double (required), 'int_memory_0.0': double (required), 'int_memory_1.0': double (required), 'int_memory_3.0': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

In [ ]:
## Залогирум production run без метрик
RUN_NAME = "production_model"

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "stage": "production",
            "trained_on_full_dataset": True,
            "source_model": "sklearn_fe + mlxtend_sfs",
            "n_estimators": 100,
            "random_state": 42,
            "selected_features_count": len(selected_feature_idx),
        }
    )

    mlflow.log_artifact(str(production_feature_names_path))
    mlflow.log_artifact(str(selected_feature_idx_path))
    mlflow.log_artifact(str(selected_feature_names_path))
    mlflow.log_artifact(str(PROJECT_ROOT / "requirements.txt"))

    mlflow.sklearn.log_model(
        sk_model=production_model,
        artifact_path="model",
        signature=production_signature,
        input_example=production_input_example,
        registered_model_name="mobile_price_classifier",
    )

    production_run_id = run.info.run_id

print("Production run logged to MLflow")
print("Run ID:", production_run_id)

Registered model 'mobile_price_classifier' already exists. Creating a new version of this model...
2026/03/09 15:38:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mobile_price_classifier, version 6
Created version '6' of model 'mobile_price_classifier'.


2026/03/09 15:38:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run production_model at: http://127.0.0.1:5000/#/experiments/188674586182771687/runs/8a53557b6bb24b9681270b3a8a2f70bc.
2026/03/09 15:38:12 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/188674586182771687.


Production run logged to MLflow
Run ID: 8a53557b6bb24b9681270b3a8a2f70bc


In [ ]:
## Поставим тег Production версии модели
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "mobile_price_classifier"

latest_versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(latest_versions, key=lambda mv: int(mv.version))

print("Latest model version:", latest_version.version)

client.set_model_version_tag(
    name=model_name, version=latest_version.version, key="Production", value="true"
)

print(
    f"Tag 'Production=true' set for model {model_name}, version {latest_version.version}"
)

Latest model version: 6
Tag 'Production=true' set for model mobile_price_classifier, version 6


In [72]:
import mlflow

mlflow.set_tag("mlflow.sklearn_feature_engineering_random_forest", "baseline_model")